# NB01：體積量測儀器校正（Volumetric Calibration）— 秤重法

**食品分析實驗 週次 1｜Nielsen's Food Analysis, 6th ed. 對應章節：分析數據處理（Evaluating Analytical Data）**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用「秤重法」（gravimetric method）校正 10 mL 移液管（volumetric pipette）與 50 mL 滴定管（burette）的實際遞送體積。
2. 應用水的密度公式（Tanaka et al., 2001）與空氣浮力校正（air-buoyancy correction），把「表觀質量」換算成「真實體積」。
3. 計算平均值（mean）、標準差（SD）、相對標準差（RSD%）、95% 信賴區間（95% CI，t 分布）。
4. 判斷數據的有效位數（significant figures），並依 CI 決定平均值該報告到第幾位小數。
5. 區分準確度（accuracy）與精密度（precision），並依 Class A 公差判斷儀器/操作是否合格。
6. 理解系統誤差（systematic error）與隨機誤差（random error）的差別。

## 如何使用本筆記本

- 直接「全部執行」（Run all / 執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬教學資料**。
- 若要換成你們班上實際量測的資料，`load_data()` 函式支援三種資料來源：
  1. **本機/雲端硬碟路徑**：`load_data("data/nb01_sample.csv")` 或你自己上傳的檔名。
  2. **已發布的 Google 試算表 CSV 連結**：檔案 → 共用 → 發布到網路 → 選 CSV，取得類似
     `https://docs.google.com/spreadsheets/d/e/xxx/pub?output=csv` 的網址，傳入 `load_data(那個網址)`。
  3. **在 Colab 上傳檔案**：執行 Setup 區塊中被註解掉的上傳程式碼（使用 `google.colab.files.upload()`）。
  4. 若以上都沒有提供，`load_data()` 會自動退回使用本筆記本內嵌的模擬資料字串，確保筆記本在任何環境都能獨立執行。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值，僅用於練習統計計算與繪圖。做作業時請換成你們班的實際數據。

## 資料格式 (Data Schema) — 之後所有週次筆記本共用

本課程所有週次的原始資料一律採用「長格式」（long format），欄位固定如下：

| 欄位 | 說明 |
|---|---|
| `group` | 組別（1–5） |
| `student` | 學號/代號（例：`G1_S1` 代表第 1 組第 1 位同學） |
| `item` | 量測項目代號。本週：`pipette_10mL`（10 mL 移液管）、`burette_25mL`（滴定管遞送 25.00 mL）|
| `rep` | 重複次數（1–10） |
| `value` | 量測值（本週為秤重讀數，即水的表觀質量，單位見 `unit`）|
| `unit` | `value` 的單位（本週固定為 `g`）|
| `temp_C` | 量測當下的水溫（°C），用於密度校正 |
| `note` | 備註欄，如：操作異常、疑似記錄錯誤等（可空白）|

之後每週的筆記本都會把資料整理成這個長格式，方便重複使用同一套統計函式。

## 1. 環境設定 (Setup)

匯入需要的套件，並定義 `load_data()`：一個可同時在 Colab 與本機執行、且不需要額外檔案就能跑的資料載入函式（內建模擬資料以字串形式嵌入本筆記本）。

**注意**：所有圖表的座標軸標籤與標題一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼（tofu）。文字說明則使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# This is the exact content of data/nb01_sample.csv, embedded so the
# notebook is self-contained (works even without the data/ folder present).
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note
1,G1_S1,pipette_10mL,1,9.9608,g,24.2,
1,G1_S1,burette_25mL,1,24.9056,g,25.5,
1,G1_S1,pipette_10mL,2,9.9624,g,24.3,
1,G1_S1,burette_25mL,2,24.9293,g,24.7,
1,G1_S1,pipette_10mL,3,9.9684,g,25.3,
1,G1_S1,burette_25mL,3,24.9003,g,24.9,
1,G1_S1,pipette_10mL,4,9.96,g,25.1,
1,G1_S1,burette_25mL,4,24.9088,g,25.7,
1,G1_S1,pipette_10mL,5,9.9601,g,25.5,
1,G1_S1,burette_25mL,5,24.9057,g,25.9,
1,G1_S1,pipette_10mL,6,9.9613,g,25.6,
1,G1_S1,burette_25mL,6,24.921,g,24.9,
1,G1_S1,pipette_10mL,7,9.968,g,24.3,
1,G1_S1,burette_25mL,7,24.9041,g,25.5,
1,G1_S1,pipette_10mL,8,9.9616,g,24.7,
1,G1_S1,burette_25mL,8,24.934,g,24.9,
1,G1_S1,pipette_10mL,9,9.9625,g,24.3,
1,G1_S1,burette_25mL,9,24.9284,g,24.5,
1,G1_S1,pipette_10mL,10,9.9669,g,24.9,
1,G1_S1,burette_25mL,10,24.9136,g,25.7,
1,G1_S2,pipette_10mL,1,9.96,g,24.4,
1,G1_S2,burette_25mL,1,24.9105,g,25.6,
1,G1_S2,pipette_10mL,2,9.9559,g,25.4,
1,G1_S2,burette_25mL,2,24.9006,g,24.9,
1,G1_S2,pipette_10mL,3,9.9657,g,24.3,
1,G1_S2,burette_25mL,3,24.9163,g,25.3,
1,G1_S2,pipette_10mL,4,9.9566,g,25.1,
1,G1_S2,burette_25mL,4,24.9119,g,25.3,
1,G1_S2,pipette_10mL,5,9.961,g,25.1,
1,G1_S2,burette_25mL,5,24.9076,g,24.1,
1,G1_S2,pipette_10mL,6,9.959,g,24.4,
1,G1_S2,burette_25mL,6,24.8898,g,25.7,
1,G1_S2,pipette_10mL,7,9.9624,g,24.1,
1,G1_S2,burette_25mL,7,24.9176,g,24.6,
1,G1_S2,pipette_10mL,8,9.9585,g,25.1,
1,G1_S2,burette_25mL,8,24.9018,g,25.3,
1,G1_S2,pipette_10mL,9,9.9541,g,25.6,
1,G1_S2,burette_25mL,9,24.9092,g,24.0,
1,G1_S2,pipette_10mL,10,9.959,g,25.4,
1,G1_S2,burette_25mL,10,24.9118,g,24.3,
1,G1_S3,pipette_10mL,1,9.9551,g,24.6,
1,G1_S3,burette_25mL,1,24.8798,g,24.7,
1,G1_S3,pipette_10mL,2,9.9514,g,24.2,
1,G1_S3,burette_25mL,2,24.8801,g,25.8,
1,G1_S3,pipette_10mL,3,9.9623,g,24.5,
1,G1_S3,burette_25mL,3,24.904,g,25.6,
1,G1_S3,pipette_10mL,4,9.9537,g,24.9,
1,G1_S3,burette_25mL,4,24.8982,g,24.2,
1,G1_S3,pipette_10mL,5,9.089,g,24.9,possible data-entry error (數據疑似誤植)
1,G1_S3,burette_25mL,5,24.9036,g,24.6,
1,G1_S3,pipette_10mL,6,9.9486,g,24.4,
1,G1_S3,burette_25mL,6,24.8873,g,25.5,
1,G1_S3,pipette_10mL,7,9.9548,g,24.9,
1,G1_S3,burette_25mL,7,24.8932,g,25.2,
1,G1_S3,pipette_10mL,8,9.9574,g,24.2,
1,G1_S3,burette_25mL,8,24.9222,g,24.1,
1,G1_S3,pipette_10mL,9,9.957,g,24.7,
1,G1_S3,burette_25mL,9,24.8865,g,24.2,
1,G1_S3,pipette_10mL,10,9.9515,g,24.3,
1,G1_S3,burette_25mL,10,24.8776,g,25.2,
2,G2_S1,pipette_10mL,1,9.9624,g,25.6,
2,G2_S1,burette_25mL,1,24.9612,g,25.0,
2,G2_S1,pipette_10mL,2,9.9561,g,25.9,
2,G2_S1,burette_25mL,2,24.9228,g,24.9,
2,G2_S1,pipette_10mL,3,9.9609,g,24.7,
2,G2_S1,burette_25mL,3,24.9168,g,24.9,
2,G2_S1,pipette_10mL,4,9.9601,g,25.7,
2,G2_S1,burette_25mL,4,24.9074,g,24.3,
2,G2_S1,pipette_10mL,5,9.962,g,24.2,
2,G2_S1,burette_25mL,5,24.9249,g,24.6,
2,G2_S1,pipette_10mL,6,9.9605,g,25.5,
2,G2_S1,burette_25mL,6,24.9159,g,24.2,
2,G2_S1,pipette_10mL,7,9.9597,g,24.5,
2,G2_S1,burette_25mL,7,24.949,g,25.1,
2,G2_S1,pipette_10mL,8,9.9622,g,25.7,
2,G2_S1,burette_25mL,8,24.9345,g,24.6,
2,G2_S1,pipette_10mL,9,9.962,g,24.6,
2,G2_S1,burette_25mL,9,24.9117,g,24.5,
2,G2_S1,pipette_10mL,10,9.9618,g,24.3,
2,G2_S1,burette_25mL,10,24.9,g,24.9,
2,G2_S2,pipette_10mL,1,9.973,g,25.0,
2,G2_S2,burette_25mL,1,24.9045,g,25.5,
2,G2_S2,pipette_10mL,2,9.9692,g,24.7,
2,G2_S2,burette_25mL,2,24.9157,g,25.5,
2,G2_S2,pipette_10mL,3,9.9567,g,25.9,
2,G2_S2,burette_25mL,3,24.9531,g,24.2,
2,G2_S2,pipette_10mL,4,9.966,g,24.3,
2,G2_S2,burette_25mL,4,24.8933,g,25.2,
2,G2_S2,pipette_10mL,5,9.9706,g,24.6,
2,G2_S2,burette_25mL,5,24.9095,g,25.9,
2,G2_S2,pipette_10mL,6,9.9698,g,24.3,
2,G2_S2,burette_25mL,6,24.9204,g,24.5,
2,G2_S2,pipette_10mL,7,9.9545,g,25.4,
2,G2_S2,burette_25mL,7,24.8996,g,25.0,
2,G2_S2,pipette_10mL,8,9.9611,g,25.2,
2,G2_S2,burette_25mL,8,24.9289,g,25.6,
2,G2_S2,pipette_10mL,9,9.9651,g,25.5,
2,G2_S2,burette_25mL,9,24.9298,g,24.2,
2,G2_S2,pipette_10mL,10,9.965,g,24.8,
2,G2_S2,burette_25mL,10,24.9257,g,25.0,
2,G2_S3,pipette_10mL,1,9.9677,g,25.1,
2,G2_S3,burette_25mL,1,24.9228,g,24.2,
2,G2_S3,pipette_10mL,2,9.9774,g,24.8,
2,G2_S3,burette_25mL,2,24.9069,g,25.2,
2,G2_S3,pipette_10mL,3,9.9601,g,25.6,
2,G2_S3,burette_25mL,3,24.9162,g,25.6,
2,G2_S3,pipette_10mL,4,9.9665,g,24.2,
2,G2_S3,burette_25mL,4,24.8911,g,25.6,
2,G2_S3,pipette_10mL,5,9.9629,g,25.1,
2,G2_S3,burette_25mL,5,24.9349,g,25.7,
2,G2_S3,pipette_10mL,6,9.9644,g,24.8,
2,G2_S3,burette_25mL,6,24.897,g,25.3,
2,G2_S3,pipette_10mL,7,9.9757,g,24.9,
2,G2_S3,burette_25mL,7,24.9088,g,24.5,
2,G2_S3,pipette_10mL,8,9.9768,g,25.6,
2,G2_S3,burette_25mL,8,24.9188,g,24.1,
2,G2_S3,pipette_10mL,9,9.9757,g,24.3,
2,G2_S3,burette_25mL,9,24.9169,g,24.6,
2,G2_S3,pipette_10mL,10,9.9668,g,25.8,
2,G2_S3,burette_25mL,10,24.918,g,24.6,
3,G3_S1,pipette_10mL,1,9.9338,g,24.1,
3,G3_S1,burette_25mL,1,24.8693,g,25.4,
3,G3_S1,pipette_10mL,2,9.9313,g,24.6,
3,G3_S1,burette_25mL,2,24.8564,g,25.8,
3,G3_S1,pipette_10mL,3,9.9321,g,24.1,
3,G3_S1,burette_25mL,3,24.8916,g,24.5,
3,G3_S1,pipette_10mL,4,9.9283,g,24.8,
3,G3_S1,burette_25mL,4,24.875,g,24.7,
3,G3_S1,pipette_10mL,5,9.9357,g,24.2,
3,G3_S1,burette_25mL,5,24.8814,g,24.1,
3,G3_S1,pipette_10mL,6,9.9349,g,24.2,
3,G3_S1,burette_25mL,6,24.8519,g,25.8,
3,G3_S1,pipette_10mL,7,9.9293,g,25.6,
3,G3_S1,burette_25mL,7,24.8647,g,25.3,
3,G3_S1,pipette_10mL,8,9.9311,g,24.3,
3,G3_S1,burette_25mL,8,24.8873,g,25.0,
3,G3_S1,pipette_10mL,9,9.9296,g,24.9,
3,G3_S1,burette_25mL,9,24.8716,g,25.3,
3,G3_S1,pipette_10mL,10,9.9265,g,24.3,
3,G3_S1,burette_25mL,10,24.8915,g,24.8,
3,G3_S2,pipette_10mL,1,9.9555,g,25.4,
3,G3_S2,burette_25mL,1,24.8952,g,25.9,
3,G3_S2,pipette_10mL,2,9.9627,g,25.0,
3,G3_S2,burette_25mL,2,24.9164,g,25.1,
3,G3_S2,pipette_10mL,3,9.9605,g,25.3,
3,G3_S2,burette_25mL,3,24.8933,g,25.1,
3,G3_S2,pipette_10mL,4,9.9629,g,24.6,
3,G3_S2,burette_25mL,4,24.9076,g,24.4,
3,G3_S2,pipette_10mL,5,9.9563,g,25.7,
3,G3_S2,burette_25mL,5,24.9021,g,25.2,
3,G3_S2,pipette_10mL,6,9.9596,g,25.5,
3,G3_S2,burette_25mL,6,24.8863,g,25.9,
3,G3_S2,pipette_10mL,7,9.9534,g,25.9,
3,G3_S2,burette_25mL,7,24.9005,g,25.2,
3,G3_S2,pipette_10mL,8,9.9522,g,25.6,
3,G3_S2,burette_25mL,8,24.8795,g,24.5,
3,G3_S2,pipette_10mL,9,9.9612,g,24.2,
3,G3_S2,burette_25mL,9,24.9008,g,25.1,
3,G3_S2,pipette_10mL,10,9.9594,g,24.9,
3,G3_S2,burette_25mL,10,24.8955,g,25.5,
3,G3_S3,pipette_10mL,1,9.9689,g,24.2,
3,G3_S3,burette_25mL,1,24.9246,g,25.3,
3,G3_S3,pipette_10mL,2,9.9624,g,25.5,
3,G3_S3,burette_25mL,2,24.9216,g,24.7,
3,G3_S3,pipette_10mL,3,9.9598,g,25.5,
3,G3_S3,burette_25mL,3,24.9174,g,24.7,
3,G3_S3,pipette_10mL,4,9.9635,g,24.3,
3,G3_S3,burette_25mL,4,24.8994,g,25.9,
3,G3_S3,pipette_10mL,5,9.9674,g,24.8,
3,G3_S3,burette_25mL,5,24.9073,g,25.1,
3,G3_S3,pipette_10mL,6,9.9623,g,25.6,
3,G3_S3,burette_25mL,6,24.9332,g,24.8,
3,G3_S3,pipette_10mL,7,9.9628,g,25.4,
3,G3_S3,burette_25mL,7,24.9481,g,24.2,
3,G3_S3,pipette_10mL,8,9.9495,g,25.3,
3,G3_S3,burette_25mL,8,24.9107,g,25.2,
3,G3_S3,pipette_10mL,9,9.9686,g,24.0,
3,G3_S3,burette_25mL,9,24.8986,g,25.7,
3,G3_S3,pipette_10mL,10,9.9563,g,24.9,
3,G3_S3,burette_25mL,10,24.9255,g,24.6,
4,G4_S1,pipette_10mL,1,9.967,g,24.6,
4,G4_S1,burette_25mL,1,24.9274,g,24.2,
4,G4_S1,pipette_10mL,2,9.9672,g,24.2,
4,G4_S1,burette_25mL,2,24.9122,g,24.1,
4,G4_S1,pipette_10mL,3,9.9669,g,24.3,
4,G4_S1,burette_25mL,3,24.9162,g,24.4,
4,G4_S1,pipette_10mL,4,9.9628,g,25.3,
4,G4_S1,burette_25mL,4,24.9416,g,24.0,
4,G4_S1,pipette_10mL,5,9.9596,g,25.2,
4,G4_S1,burette_25mL,5,24.8792,g,24.5,
4,G4_S1,pipette_10mL,6,9.9692,g,24.1,
4,G4_S1,burette_25mL,6,24.9176,g,25.9,
4,G4_S1,pipette_10mL,7,9.9678,g,25.4,
4,G4_S1,burette_25mL,7,24.911,g,24.1,
4,G4_S1,pipette_10mL,8,9.9662,g,24.2,
4,G4_S1,burette_25mL,8,24.9182,g,25.5,
4,G4_S1,pipette_10mL,9,9.9607,g,25.7,
4,G4_S1,burette_25mL,9,24.8999,g,25.5,
4,G4_S1,pipette_10mL,10,9.9684,g,24.5,
4,G4_S1,burette_25mL,10,24.8873,g,26.0,
4,G4_S2,pipette_10mL,1,9.9563,g,25.4,
4,G4_S2,burette_25mL,1,24.8767,g,24.2,
4,G4_S2,pipette_10mL,2,9.9471,g,26.0,
4,G4_S2,burette_25mL,2,24.8743,g,25.6,
4,G4_S2,pipette_10mL,3,9.9544,g,25.1,
4,G4_S2,burette_25mL,3,24.8694,g,25.9,
4,G4_S2,pipette_10mL,4,9.9575,g,25.2,
4,G4_S2,burette_25mL,4,24.9007,g,25.3,
4,G4_S2,pipette_10mL,5,9.954,g,24.3,
4,G4_S2,burette_25mL,5,24.9196,g,24.8,
4,G4_S2,pipette_10mL,6,9.9552,g,24.2,
4,G4_S2,burette_25mL,6,24.9045,g,24.6,
4,G4_S2,pipette_10mL,7,9.9547,g,24.9,
4,G4_S2,burette_25mL,7,24.8857,g,25.3,
4,G4_S2,pipette_10mL,8,9.9457,g,24.8,
4,G4_S2,burette_25mL,8,24.8595,g,25.2,
4,G4_S2,pipette_10mL,9,9.9537,g,24.1,
4,G4_S2,burette_25mL,9,24.9031,g,24.9,
4,G4_S2,pipette_10mL,10,9.9543,g,25.1,
4,G4_S2,burette_25mL,10,24.8738,g,25.4,
4,G4_S3,pipette_10mL,1,9.9628,g,24.9,
4,G4_S3,burette_25mL,1,24.9244,g,24.7,
4,G4_S3,pipette_10mL,2,9.9629,g,25.5,
4,G4_S3,burette_25mL,2,24.9034,g,25.8,
4,G4_S3,pipette_10mL,3,9.9469,g,25.0,
4,G4_S3,burette_25mL,3,24.8715,g,24.6,
4,G4_S3,pipette_10mL,4,9.9746,g,25.0,
4,G4_S3,burette_25mL,4,24.9711,g,24.1,
4,G4_S3,pipette_10mL,5,9.959,g,24.1,
4,G4_S3,burette_25mL,5,24.8979,g,24.7,
4,G4_S3,pipette_10mL,6,9.9269,g,24.6,
4,G4_S3,burette_25mL,6,24.8185,g,24.0,
4,G4_S3,pipette_10mL,7,9.9877,g,25.7,
4,G4_S3,burette_25mL,7,24.88,g,24.3,
4,G4_S3,pipette_10mL,8,9.9054,g,24.2,
4,G4_S3,burette_25mL,8,25.026,g,25.9,
4,G4_S3,pipette_10mL,9,9.95,g,25.5,
4,G4_S3,burette_25mL,9,24.8311,g,25.4,
4,G4_S3,pipette_10mL,10,9.9649,g,24.8,
4,G4_S3,burette_25mL,10,24.9571,g,24.0,
5,G5_S1,pipette_10mL,1,9.9544,g,26.0,
5,G5_S1,burette_25mL,1,24.9065,g,25.7,
5,G5_S1,pipette_10mL,2,9.967,g,24.3,
5,G5_S1,burette_25mL,2,24.9145,g,24.8,
5,G5_S1,pipette_10mL,3,9.96,g,25.5,
5,G5_S1,burette_25mL,3,24.8807,g,24.9,
5,G5_S1,pipette_10mL,4,9.9603,g,24.4,
5,G5_S1,burette_25mL,4,24.8637,g,24.1,
5,G5_S1,pipette_10mL,5,9.9595,g,24.6,
5,G5_S1,burette_25mL,5,24.8416,g,25.2,
5,G5_S1,pipette_10mL,6,9.9565,g,24.2,
5,G5_S1,burette_25mL,6,24.8866,g,25.7,
5,G5_S1,pipette_10mL,7,9.9587,g,25.3,
5,G5_S1,burette_25mL,7,24.8871,g,25.5,
5,G5_S1,pipette_10mL,8,9.9636,g,24.7,
5,G5_S1,burette_25mL,8,24.8956,g,24.7,
5,G5_S1,pipette_10mL,9,9.9614,g,25.0,
5,G5_S1,burette_25mL,9,24.8805,g,25.7,
5,G5_S1,pipette_10mL,10,9.9557,g,25.8,
5,G5_S1,burette_25mL,10,24.8593,g,25.7,
5,G5_S2,pipette_10mL,1,9.9581,g,25.8,
5,G5_S2,burette_25mL,1,24.9005,g,24.4,
5,G5_S2,pipette_10mL,2,9.9607,g,24.4,
5,G5_S2,burette_25mL,2,24.886,g,25.9,
5,G5_S2,pipette_10mL,3,9.9543,g,24.6,
5,G5_S2,burette_25mL,3,24.8941,g,24.0,
5,G5_S2,pipette_10mL,4,9.9598,g,25.4,
5,G5_S2,burette_25mL,4,24.8963,g,26.0,
5,G5_S2,pipette_10mL,5,9.9588,g,25.0,
5,G5_S2,burette_25mL,5,24.8454,g,25.8,
5,G5_S2,pipette_10mL,6,9.9563,g,25.9,
5,G5_S2,burette_25mL,6,24.8896,g,25.7,
5,G5_S2,pipette_10mL,7,9.9533,g,25.5,
5,G5_S2,burette_25mL,7,24.8868,g,25.1,
5,G5_S2,pipette_10mL,8,9.952,g,25.9,
5,G5_S2,burette_25mL,8,24.9121,g,24.6,
5,G5_S2,pipette_10mL,9,9.9567,g,25.5,
5,G5_S2,burette_25mL,9,24.8714,g,24.2,
5,G5_S2,pipette_10mL,10,9.956,g,25.6,
5,G5_S2,burette_25mL,10,24.881,g,26.0,
5,G5_S3,pipette_10mL,1,9.9647,g,25.1,
5,G5_S3,burette_25mL,1,24.9147,g,25.8,
5,G5_S3,pipette_10mL,2,9.9628,g,24.1,
5,G5_S3,burette_25mL,2,24.9061,g,24.1,
5,G5_S3,pipette_10mL,3,9.9606,g,25.4,
5,G5_S3,burette_25mL,3,24.8997,g,25.5,
5,G5_S3,pipette_10mL,4,9.9693,g,25.5,
5,G5_S3,burette_25mL,4,24.9107,g,24.1,
5,G5_S3,pipette_10mL,5,9.9652,g,25.2,
5,G5_S3,burette_25mL,5,24.9208,g,24.0,
5,G5_S3,pipette_10mL,6,9.9685,g,25.4,
5,G5_S3,burette_25mL,6,24.9085,g,24.8,
5,G5_S3,pipette_10mL,7,9.9624,g,25.3,
5,G5_S3,burette_25mL,7,24.9245,g,25.7,
5,G5_S3,pipette_10mL,8,9.9641,g,25.7,
5,G5_S3,burette_25mL,8,24.907,g,25.2,
5,G5_S3,pipette_10mL,9,9.9681,g,24.5,
5,G5_S3,burette_25mL,9,24.9217,g,24.4,
5,G5_S3,pipette_10mL,10,9.9621,g,24.7,
5,G5_S3,burette_25mL,10,24.9299,g,25.6,
"""

In [ ]:
def load_data(source=None):
    """Load NB01 calibration data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note

    Parameters
    ----------
    source : str or None
        - None: try a local "data/nb01_sample.csv" relative path first,
          otherwise fall back to the embedded SAMPLE_CSV string.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link): read
          directly with pandas.read_csv.

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb01_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['student'].nunique()} 位學生、"
      f"{df['group'].nunique()} 組、{df['item'].nunique()} 個量測項目。")
df.head()

### （選用）在 Colab 上傳班級實際資料

若在 Google Colab 中執行、且要換成全班實際量測值，取消下方程式碼的註解後執行即可；在非 Colab 環境（例如本機 Jupyter）執行這個 cell 不會有任何作用。

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 水的密度與空氣浮力校正 (Water Density & Air-Buoyancy Correction)

秤重法校正體積的核心概念：**用天平量到的「表觀質量」(apparent mass) 換算回「真實體積」**，中間需要兩個校正：

### (1) 水的密度會隨溫度變化

本筆記本採用 Tanaka et al. (2001) 提出的水密度公式：

$$\rho_w(t) = a_5 \left[1 - \frac{(t + a_1)^2 (t + a_2)}{a_3 (t + a_4)}\right] \quad (\text{kg/m}^3)$$

其中 $t$ 為攝氏溫度，係數為：

$a_1 = -3.983035\ °C,\quad a_2 = 301.797\ °C,\quad a_3 = 522528.9\ °C^2,\quad a_4 = 69.34881\ °C,\quad a_5 = 999.974950\ \text{kg/m}^3$

在 20–25°C 之間，水溫每升高 1°C，密度大約下降 0.02%——聽起來很小，但因為它直接影響「質量→體積」的換算，仍然是校正時必須考慮的項目。

### (2) 空氣浮力校正 (Air-Buoyancy Correction)

天平顯示的「表觀質量」$m_{app}$ 其實同時受到「水」與「砝碼」在空氣中受到的浮力影響，兩者不會完全抵銷，因此需要校正：

$$m_{true} = m_{app} \times \left(1 + \rho_{air}\left(\frac{1}{\rho_w} - \frac{1}{\rho_{weights}}\right)\right)$$

其中 $\rho_{air} \approx 0.0012\ \text{g/mL}$（室溫空氣密度）、$\rho_{weights} \approx 8.0\ \text{g/mL}$（砝碼密度，不鏽鋼材質的常用近似值）。

這個校正的量級大約是 **0.1%**（$\rho_{air}\left(\frac{1}{\rho_w}-\frac{1}{\rho_{weights}}\right) \approx 0.00105$），雖然不大，但對於公差只有 ±0.02 mL（相對誤差 0.2%）的 Class A 移液管來說，是不可忽略的系統性修正。

最後，真實體積為：

$$V = \frac{m_{true}}{\rho_w}$$

In [ ]:
TANAKA_A1 = -3.983035
TANAKA_A2 = 301.797
TANAKA_A3 = 522528.9
TANAKA_A4 = 69.34881
TANAKA_A5 = 999.974950

RHO_AIR = 0.0012       # g/mL, typical lab air density
RHO_WEIGHTS = 8.0      # g/mL, approx. density of calibration weights


def water_density_tanaka(temp_C):
    """Tanaka et al. (2001) water density equation. Returns density in g/mL."""
    t = np.asarray(temp_C, dtype=float)
    rho_kg_m3 = TANAKA_A5 * (
        1.0 - ((t + TANAKA_A1) ** 2 * (t + TANAKA_A2)) / (TANAKA_A3 * (t + TANAKA_A4))
    )
    return rho_kg_m3 / 1000.0  # kg/m^3 -> g/mL


def buoyancy_factor(rho_water, rho_air=RHO_AIR, rho_weights=RHO_WEIGHTS):
    return 1.0 + rho_air * (1.0 / rho_water - 1.0 / rho_weights)


def apparent_to_true_mass(m_app, rho_water, rho_air=RHO_AIR, rho_weights=RHO_WEIGHTS):
    return m_app * buoyancy_factor(rho_water, rho_air, rho_weights)


def volume_from_mass(m_app, temp_C):
    """Convert an apparent (on-balance) mass of water at temp_C into the
    true delivered volume (mL), applying density + buoyancy corrections."""
    rho_w = water_density_tanaka(temp_C)
    m_true = apparent_to_true_mass(m_app, rho_w)
    return m_true / rho_w

In [ ]:
# Density table, 20-30 C
temps = np.arange(20, 31, 1)
density_table = pd.DataFrame({
    "temp_C": temps,
    "density_g_per_mL": water_density_tanaka(temps),
    "buoyancy_factor": buoyancy_factor(water_density_tanaka(temps)),
})
density_table

In [ ]:
# Sanity check: an apparent mass of 9.9700 g at 25 C should correspond to
# a delivered volume close to the 10.00 mL nominal pipette volume.
m_app_check = 9.9700
temp_check = 25.0
rho_check = water_density_tanaka(temp_check)
factor_check = buoyancy_factor(rho_check)
V_check = volume_from_mass(m_app_check, temp_check)

print(f"rho_w(25C)       = {rho_check:.6f} g/mL   (expect ~0.997047)")
print(f"buoyancy factor  = {factor_check:.6f}      (expect ~1.00105)")
print(f"V from 9.9700 g  = {V_check:.4f} mL        (expect close to 10.00 mL)")
print(f"deviation        = {(V_check - 10.00):+.4f} mL  ({(V_check/10.00 - 1)*100:+.3f}%)")

## 3. 逐生統計量 (Per-Student Statistics)

先把每一筆秤重數據換算成體積，再依「組別＋學生＋量測項目」分組，計算：

- **n**：重複次數
- **mean**：平均體積
- **SD**：標準差（樣本標準差，$n-1$）
- **RSD%**：相對標準差 $= \dfrac{SD}{mean}\times 100\%$，用來比較不同尺度量測的精密度
- **95% CI**：以 t 分布計算的 95% 信賴區間半寬，$CI = t_{0.975,\,n-1} \times \dfrac{SD}{\sqrt{n}}$
- **bias**：平均值與標稱值（nominal）的差，$bias = mean - nominal$，反映系統誤差

In [ ]:
NOMINAL_VOLUME = {
    "pipette_10mL": 10.00,
    "burette_25mL": 25.00,
}

df["volume_mL"] = [
    volume_from_mass(v, t) for v, t in zip(df["value"], df["temp_C"])
]


def summarize_student(g):
    n = len(g)
    mean = g["volume_mL"].mean()
    sd = g["volume_mL"].std(ddof=1)
    rsd_pct = sd / mean * 100
    sem = sd / np.sqrt(n)
    t_crit = stats.t.ppf(0.975, df=n - 1)
    ci95 = t_crit * sem
    item = g.name[2]  # g.name = (group, student, item)
    nominal = NOMINAL_VOLUME[item]
    bias = mean - nominal
    return pd.Series({
        "n": n, "mean_mL": mean, "sd_mL": sd, "rsd_pct": rsd_pct,
        "ci95_mL": ci95, "nominal_mL": nominal, "bias_mL": bias,
    })


stats_df = (
    df.groupby(["group", "student", "item"])
    .apply(summarize_student)
    .reset_index()
)
stats_df

## 4. 與 Class A 公差比較 (Comparison with Class A Tolerance)

依 ASTM 標準的 Class A 容量玻璃器皿公差：

- **10 mL 移液管**：±0.02 mL（ASTM E969，Class A volumetric pipette tolerance）
- **50 mL 滴定管**：±0.05 mL（ASTM E287，Class A burette tolerance；此公差是針對**整支滴定管**的遞送誤差，不論在哪個刻度區間遞送都適用同一個公差值）

### 準確度 vs 精密度的判斷準則（教學簡化版）

- **準確 (accurate)**：$|bias| \le$ 公差
- **精密 (precise)**：$SD \le$ 公差 / 2 （這是本課程給的教學簡化門檻，讓分類有一個明確、可重現的切點；實務上精密度的可接受標準會依實驗室 SOP 而定）

把兩個判準交叉，可以得到 2×2 分類：

In [ ]:
TOLERANCE = {
    "pipette_10mL": 0.02,
    "burette_25mL": 0.05,
}

stats_df["tolerance_mL"] = stats_df["item"].map(TOLERANCE)
stats_df["is_accurate"] = stats_df["bias_mL"].abs() <= stats_df["tolerance_mL"]
stats_df["is_precise"] = stats_df["sd_mL"] <= (stats_df["tolerance_mL"] / 2)


def classify(row):
    if row["is_accurate"] and row["is_precise"]:
        return "準確且精密 (accurate & precise)"
    if row["is_accurate"] and not row["is_precise"]:
        return "準確但不精密 (accurate, not precise)"
    if not row["is_accurate"] and row["is_precise"]:
        return "精密但不準確 (precise, not accurate)"
    return "不準確且不精密 (neither)"


stats_df["category"] = stats_df.apply(classify, axis=1)
stats_df[["group", "student", "item", "mean_mL", "bias_mL", "sd_mL", "category"]]

In [ ]:
print(stats_df.groupby(["item", "category"]).size())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, item in zip(axes, ["pipette_10mL", "burette_25mL"]):
    sub = stats_df[stats_df["item"] == item]
    tol = TOLERANCE[item]
    precision_threshold = tol / 2

    ax.axvspan(-tol, tol, color="tab:green", alpha=0.12, label="Accuracy tolerance")
    ax.axhspan(0, precision_threshold, color="tab:blue", alpha=0.10, label="Precision threshold")
    ax.axvline(0, color="grey", linewidth=1)

    ax.scatter(sub["bias_mL"], sub["sd_mL"], s=60, color="tab:red", zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(r["student"], (r["bias_mL"], r["sd_mL"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)

    ax.set_xlabel("Bias (mL)  =  mean - nominal")
    ax.set_ylabel("SD (mL)")
    ax.set_title(f"Target diagram: {item}")
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## 5. 視覺化 (Visualizations)

### 5.1 每位學生的所有重複測量值（dot/strip plot）

可以看出組內變異的大小，也能一眼認出離群值（outlier）。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=False)

for ax, item in zip(axes, ["pipette_10mL", "burette_25mL"]):
    sub = df[df["item"] == item].copy()
    students = sorted(sub["student"].unique())
    student_pos = {s: i for i, s in enumerate(students)}
    rng_jitter = np.random.default_rng(0)
    x = sub["student"].map(student_pos) + rng_jitter.uniform(-0.15, 0.15, size=len(sub))

    ax.scatter(x, sub["volume_mL"], s=18, alpha=0.7, color="tab:blue", label="reps")

    # highlight rows flagged in the note column (e.g. data-entry issues)
    flagged = sub[sub["note"].astype(str).str.len() > 0]
    if len(flagged) > 0:
        fx = flagged["student"].map(student_pos)
        ax.scatter(fx, flagged["volume_mL"], s=90, facecolors="none",
                   edgecolors="tab:red", linewidths=2, label="flagged (see note)", zorder=5)

    ax.axhline(NOMINAL_VOLUME[item], color="black", linestyle="--", linewidth=1, label="nominal")
    ax.set_xticks(range(len(students)))
    ax.set_xticklabels(students, rotation=45, ha="right")
    ax.set_ylabel("Volume (mL)")
    ax.set_title(f"All replicates: {item}")
    ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

### 5.2 平均值 ± 95% CI 長條圖

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=False)

for ax, item in zip(axes, ["pipette_10mL", "burette_25mL"]):
    sub = stats_df[stats_df["item"] == item].sort_values("student")
    ax.bar(sub["student"], sub["mean_mL"], yerr=sub["ci95_mL"], capsize=4,
           color="tab:orange", alpha=0.85)
    ax.axhline(NOMINAL_VOLUME[item], color="black", linestyle="--", linewidth=1, label="nominal")
    ax.set_ylabel("Mean volume (mL)")
    ax.set_title(f"Mean ± 95% CI: {item}")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

## 6. 有效位數 (Significant Figures)

報告「平均值 ± 不確定度」時，有一個常見規則：

1. 先把不確定度（這裡用 95% CI）四捨五入到 **1–2 位有效數字**。
2. 平均值只能報告到跟不確定度**相同的小數位數**——多報的位數沒有意義（超出量測能分辨的範圍），少報則會遺失資訊。

下面定義一個小工具函式，自動完成這個規則。

In [ ]:
from math import floor, log10


def round_to_sig_figs(x, sig_figs):
    """Round x to a given number of significant figures.
    Returns (rounded_value, decimal_places_used)."""
    if x == 0:
        return 0.0, 0
    decimals = sig_figs - int(floor(log10(abs(x)))) - 1
    return round(x, decimals), decimals


def report_mean_ci(mean, ci, sig_figs=2):
    """Format 'mean ± ci' with the CI rounded to sig_figs significant
    figures, and the mean rounded to the same decimal place."""
    ci_rounded, decimals = round_to_sig_figs(ci, sig_figs)
    decimals = max(decimals, 0)
    mean_rounded = round(mean, decimals)
    fmt = "{:." + str(decimals) + "f} ± {:." + str(decimals) + "f}"
    return fmt.format(mean_rounded, ci_rounded)

In [ ]:
# Examples
examples = stats_df.sample(4, random_state=1)
for _, r in examples.iterrows():
    text = report_mean_ci(r["mean_mL"], r["ci95_mL"])
    print(f"{r['student']:>8s}  {r['item']:<14s}  raw: {r['mean_mL']:.6f} +/- {r['ci95_mL']:.6f} mL"
          f"   ->   reported: {text} mL")

## 7. 班級層級分析 (Class-Level Analysis)

### 7.1 各組平均

把同一組的三位同學數據合併，看看組間是否有系統性差異。

In [ ]:
group_level = (
    df.groupby(["group", "item"])["volume_mL"]
    .agg(n="count", mean_mL="mean", sd_mL="std")
    .reset_index()
)
group_level

### 7.2 系統誤差 vs 隨機誤差

- **系統誤差 (systematic error)**：讓數據穩定地偏向某一邊，例如操作習慣造成移液管總是少放一點水 → 反映在 `bias_mL` 上，重複測量再多次也不會消失，只能靠改進技術或校正來修正。
- **隨機誤差 (random error)**：每次測量圍繞真值上下波動，反映在 `sd_mL` / `rsd_pct` 上，重複測量次數越多，平均值受隨機誤差影響越小（CI 會縮小）。

請對照第 4 節的 target diagram：往左右偏離中心多的同學，系統誤差較大；點分布縱向範圍大的同學，隨機誤差（精密度不佳）較明顯。

### 7.3 單樣本 t 檢定（預告）

這裡示範用單樣本 t 檢定（one-sample t-test）檢查「某位同學的平均體積是否顯著偏離標稱值」，作為之後統計課程的預告。**完整的假設檢定推導、前提假設與多重比較校正，我們會在後面章節更嚴謹地處理**，這裡先建立直覺。

In [ ]:
example_student = stats_df.iloc[0]
sample_data = df[
    (df["student"] == example_student["student"]) & (df["item"] == example_student["item"])
]["volume_mL"]

t_stat, p_value = stats.ttest_1samp(sample_data, popmean=example_student["nominal_mL"])
print(f"Student: {example_student['student']}, item: {example_student['item']}")
print(f"H0: population mean volume = {example_student['nominal_mL']:.2f} mL")
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("p < 0.05 -> 在 alpha=0.05 下，平均體積與標稱值有顯著差異（可能有系統誤差）。")
else:
    print("p >= 0.05 -> 沒有足夠證據顯示平均體積偏離標稱值。")

## 8. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算）。

### 問題 1

如果忘記做浮力校正（只做溫度密度校正），計算出來的體積會偏高還是偏低？差多少（mL 或 %）？請用 `df` 中任一筆資料實際計算比較。

In [ ]:
# TODO: 請在這裡作答

### 問題 2

找出班上「最精密但不準確」的同學（參考第 4 節的分類結果），並推測可能的操作原因。

In [ ]:
# TODO: 請在這裡作答

### 問題 3

如果某次記錄溫度時多記了 5°C（例如實際水溫是 25°C，卻記成 30°C），對換算出來的體積影響有多大（mL 及 %）？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

比較 `pipette_10mL` 與 `burette_25mL` 的整體 RSD%（可用 `stats_df` 分組平均），哪一種儀器的精密度比較好？為什麼可能會有這樣的差異？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

如果某位同學的 95% CI 涵蓋了標稱值（nominal），但他的 `bias_mL` 卻超出 Class A 公差，這是否矛盾？該如何解讀？（提示：想想 CI 和公差分別在回答什麼問題）

In [ ]:
# TODO: 請在這裡作答

## 9. 匯出結果 (Export)

把逐生統計結果存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔，不會嘗試下載（用 try/except 保護，確保兩種環境都能跑）。

In [ ]:
output_path = "nb01_student_results.csv"
stats_df.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")